# DeepSCF: a periodic polymer
In this tutorial, you will predict the electron density of a periodic C/H polymer with the pretrained **(R+S)×3** model and use that density in a one-step non-SCF calculation. A conventional SCF calculation provides the reference density, total energy, and band structure.

Use a **CPU runtime** and run the cells in order. The first code cell downloads DeepSCF and all tutorial inputs; no manual upload or model training is needed.

**Workflow:** setup → inspect the structure and inputs → reference SCF → prepare grid data → predict the density → non-SCF calculation.

*The calculation workflow was tested in Colab on 26 October 2026 with Python 3.13.16, PyTorch 2.11.0 (CPU), and sisl 0.20.0.*

## 1. Setup
### Clone the repositories
Download the fixed DeepSCF snapshot and this tutorial into `/content`. DeepSCF provides the model and modified SIESTA source; `tutorial/deepscf/01/src` contains the polymer inputs and Python helpers.

The next cell checks the snapshot and required files. Existing clone directories are reused.

In [ ]:
%%bash
set -e
cd /content
if [ ! -d DeepSCF ]; then
  git clone https://github.com/yhkimlab/DeepSCF.git
  git -C DeepSCF checkout --detach fe3c3c18052b51101e438dad2ffa06f94e77b5bb
fi

if [ ! -d tutorial ]; then
  git clone --depth 1 --branch main https://github.com/ryonggyulee/tutorial.git tutorial
fi


In [ ]:
%%bash
set -e
cd /content/DeepSCF
# Check the tutorial snapshot.
[ "$(git rev-parse HEAD)" = fe3c3c18052b51101e438dad2ffa06f94e77b5bb ]
test -s requirements.txt
test -s src/siesta-4.1-b4/Obj/gfortran-system.make
git log -1 --format='%h %cs %s'
test -s /content/tutorial/deepscf/01/src/inputs/common.fdf
test -s /content/tutorial/deepscf/01/src/predict_polymer.py
echo "DeepSCF and tutorial inputs ready"


### Install dependencies
Use Colab's Python environment. PyTorch runs the pretrained network, h5py reads the grid dataset, and sisl reads SIESTA structures and output files. Matplotlib displays the structure, density, and bands inside the notebook.

Install the packages, then import them and check their versions.

In [ ]:
%pip install -q torch h5py hydra-core==1.3.2 omegaconf==2.3.0 "sisl[viz]" matplotlib

In [ ]:
%matplotlib inline
import sys, subprocess
import torch, h5py, numpy, hydra, omegaconf
import sisl, sisl.viz

print('Python:', sys.version.split()[0], '| PyTorch:', torch.__version__, '| sisl:', sisl.__version__)

### Build and launch SIESTA
Compile the modified SIESTA bundled with DeepSCF. This version writes the additional grids needed for preprocessing and reads the predicted density for the non-SCF calculation. Compilation may take a few minutes; the full log is saved to `/content/siesta-build.log`.

In [ ]:
%%bash
set -e
apt-get update -qq
apt-get install -y -qq build-essential binutils gfortran libblas-dev liblapack-dev
cd /content/DeepSCF/src/siesta-4.1-b4/Obj
sh ../Src/obj_setup.sh
cp gfortran-system.make arch.make
# Save the build log.
make -j2 > /content/siesta-build.log 2>&1 || { tail -80 /content/siesta-build.log; exit 1; }
test -x siesta
echo "SIESTA build OK"


**Check the executable.** Launch SIESTA without an input file and inspect the version and compiler information. The missing-input error and nonzero exit code are expected here; this cell only checks that the executable starts.

In [ ]:
import subprocess, tempfile

with tempfile.TemporaryDirectory() as workdir:
    result = subprocess.run(
        ['/content/DeepSCF/src/siesta-4.1-b4/Obj/siesta'],
        stdin=subprocess.DEVNULL, capture_output=True, text=True,
        cwd=workdir, timeout=15,
    )
print(result.stdout)
print(result.stderr)
print('Exit code:', result.returncode, '(no input supplied)')


## 2. Reference SCF calculation
### Prepare the run folders
Copy the supplied inputs into separate folders for reference SCF, initial-density non-SCF, and ML-density non-SCF calculations. Each run uses a new working directory, printed below, so previous output files are not reused.

To repeat the calculation, start again from this cell and run the subsequent cells.

In [ ]:
from pathlib import Path
import sys, shutil, tempfile

ROOT = Path('/content/DeepSCF')
SRC = Path('/content/tutorial/deepscf/01/src')
PYTHON = sys.executable
SIESTA = ROOT / 'src/siesta-4.1-b4/Obj/siesta'
(ROOT / 'results').mkdir(exist_ok=True)
WORK = Path(tempfile.mkdtemp(prefix='polymer-', dir=ROOT / 'results'))
SCF, NSCF = WORK / 'scf/polymer', WORK / 'non_scf'
INITIAL = WORK / 'initial'
for folder, mode in [(SCF, 'scf'), (INITIAL, 'non_scf'), (NSCF, 'non_scf')]:
    folder.mkdir(parents=True)
    for name in ['STRUCT.fdf', 'common.fdf', 'C.psf', 'H.psf']:
        shutil.copy(SRC / 'inputs' / name, folder / name)
    shutil.copy(SRC / 'inputs' / mode / 'RUN.fdf', folder / 'RUN.fdf')
print('Working directory:', WORK)

### Inspect the polymer structure
Read the supplied `STRUCT.fdf` with sisl. It contains **24 atoms (C₈H₁₆)** in a **20 × 20 × 10 Å** cell. The polymer extends periodically along **z**; the transverse directions contain vacuum.

The figure is an **x–z projection**, so atoms separated along y may overlap. Inspect the chain and the cell boundaries before running the calculation.

In [ ]:
geometry = sisl.get_sile(str(SCF / 'STRUCT.fdf')).read_geometry()
print('Number of atoms:', len(geometry))
print('Cell vectors (Angstrom):\n', geometry.cell)
geometry.plot(axes='xz', show_cell='box', backend='matplotlib').show()

### Inspect the FDF inputs
SIESTA reads `RUN.fdf`, which includes `common.fdf`; the latter includes `STRUCT.fdf`. The files below define the structure, shared numerical settings, and SCF control.

| Input setting | Purpose in this example |
|---|---|
| `PAO.BasisSize DZP`, `XC.authors PBE` | Use a double-zeta polarized basis and PBE. |
| `kgrid_Monkhorst_Pack` | Sample the periodic chain with a 1 × 1 × 8 k-point mesh. |
| `MeshSizes 80 80 40` | Fix the grid at 0.25 Å spacing; explicit mesh sizes override the requested cutoff. |
| `MD.NumCGsteps 0` | Keep the supplied atomic positions fixed. |
| `WriteForces T` | Print atomic forces; this SIESTA version also writes `Molecular.FA` automatically. |
| `SaveRho`, `SaveIonicCharge` | Write density grids used in preprocessing. |
| `MaxSCFIterations 300`, `SCF.MustConverge T` | Require the reference SCF calculation to converge. |

`BandLines` specifies Γ → X, where **X = (0, 0, ½)** in reciprocal-lattice coordinates along the polymer axis. The bands are folded into the supplied 10 Å repeat cell.

In [ ]:
for name in ['STRUCT.fdf', 'common.fdf', 'RUN.fdf']:
    print(f'--- {name} ---')
    print((SCF / name).read_text())

### Run the reference SCF calculation
Run SIESTA with the inspected inputs. The helper checks successful completion and SCF convergence. Its detailed output is stored in `stdout.txt` and `stderr.txt` in the SCF folder.

The converged density `Molecular.RHO` and bands `Molecular.bands` will be used as references. The modified SIESTA also writes the grids consumed by DeepSCF preprocessing.

In [ ]:
subprocess.run([PYTHON, str(SRC / 'run_siesta.py'), str(SCF),
                '--siesta', str(SIESTA)], check=True)

### Inspect the reference density
Compare the initial atomic density with the converged SCF density. The plots show the density integrated along y, in e/Å², rather than a single plane through the structure.

The reported relative error uses the **full 3D grid**:
\[
\epsilon_\rho = 100\,\frac{\sum_i |\rho_i-\rho_i^{\mathrm{SCF}}|}{\sum_i \rho_i^{\mathrm{SCF}}}.
\]
The uniform grid volume cancels in this ratio. The integrated SCF charge should be close to **48 valence electrons** for C₈H₁₆.

In [ ]:
sys.path.insert(0, str(SRC))
from analysis import plot_density, plot_bands, compare_observables, plot_forces

plot_density(SCF / 'Molecular.RHO', atomic=SCF / 'rhoatom');

### Inspect the reference bands
Plot the bands near the gap along Γ–X. The energy zero is the **reference SCF valence-band maximum** on this path.

The printed gap is obtained from the sampled Γ–X eigenvalues. This is a path-based comparison, not a separate search over the full Brillouin zone.

In [ ]:
plot_bands(SCF);

## 3. Prepare the model inputs
### Convert: SIESTA grids → HDF5
Preprocessing consists of three stages: **convert → determine scaling → collect**. We run conversion first, inspect the pretrained model's scaling parameters, and then collect the normalized inputs.

`1.convert.py` reads the four SIESTA grids and writes one HDF5 file per channel. Each file contains a single dataset named `value`, with shape `(80, 80, 40)` corresponding to the x, y, and z grid dimensions.

| SIESTA file | Output under `raw/` | Contents |
|---|---|---|
| `Molecular.RHO` | `target/polymer.h5` | Converged SCF electron density |
| `Molecular.IOCH` | `feature1/polymer.h5` | Ionic charge density |
| `rhoatom` | `feature2/polymer.h5` | Initial atomic electron density |
| `rhoover` | `feature3/polymer.h5` | Orbital-overlap feature |

This step changes the file format without normalizing or resampling the grid values. The arrays retain the numerical units of the SIESTA grids; the cell and atomic coordinates are not stored in these HDF5 files. Inspect the dataset names, shapes, and value ranges below.


In [ ]:
subprocess.run([PYTHON, str(ROOT / 'tutorial/1.preprocess/1.convert.py'),
                str(WORK / 'scf'), str(WORK / 'raw')], check=True)

channels = ['target', 'feature1', 'feature2', 'feature3']
for channel in channels:
    path = WORK / 'raw' / channel / 'polymer.h5'
    with h5py.File(path, 'r') as data:
        values = data['value'][:]
        print(f'{path.relative_to(WORK)}: keys={list(data.keys())}, '
              f'shape={values.shape}, dtype={values.dtype}, '
              f'range=[{values.min():.6g}, {values.max():.6g}]')


### Scale: inspect the pretrained normalization
In the full preprocessing workflow, `2.scale.py` computes the spatial mean and standard deviation of each channel for each structure, then averages these statistics over the supplied structures:

$$
\mu_c = \frac{1}{N}\sum_{n=1}^{N}\operatorname{mean}_{\mathbf r} x_c^{(n)}(\mathbf r),
\qquad
\sigma_c = \frac{1}{N}\sum_{n=1}^{N}\operatorname{std}_{\mathbf r} x_c^{(n)}(\mathbf r).
$$

Thus, `std` is the **average of the per-structure standard deviations**, rather than the standard deviation of all grid points pooled together.

Here we use the supplied `scale_r_plus_s_x3.txt` for the pretrained **(R+S)×3** model. We do not run `2.scale.py` on this polymer: inference must use the scaling associated with the trained model.

| Entries in the scale file | Raw channel | Use in `3.collect.py` |
|---|---|---|
| `mean1`, `std1` | `target`: SCF density | Recorded statistics; target is copied unchanged |
| `mean2`, `std2` | `feature1`: ionic charge | Normalize input channel 0 |
| `mean3`, `std3` | `feature2`: atomic density | Normalize input channel 1 |
| `mean4`, `std4` | `feature3`: orbital overlap | Normalize input channel 2 |

Print the file below. The numbering includes the target first, so `feature1` uses **mean2/std2**, not mean1/std1.


In [ ]:
SCALE = ROOT / 'datasets/scale/scale_r_plus_s_x3.txt'
print(SCALE.read_text())

scale = {}
for line in SCALE.read_text().splitlines():
    if line.strip():
        key, value = line.split('=')
        scale[key.strip()] = float(value)


### Collect: normalize and assemble the dataset
`3.collect.py` applies a fixed affine transformation to each input channel:

$$
\widetilde{x}_c(\mathbf r) =
\frac{x_c(\mathbf r)-\mu_c}{\sigma_c}.
$$

The three normalized arrays are stacked along a new channel axis and saved together with the unmodified SCF target in `dataset/polymer.h5`.

| Dataset | Shape | Contents |
|---|---|---|
| `feature` | `(3, 80, 80, 40)` | Normalized ionic charge, atomic density, and orbital overlap, in that order |
| `target` | `(80, 80, 40)` | SCF reference density, without normalization |

Only `feature` enters the network. The prediction helper converts it to float32 and adds a batch dimension, giving `(1, 3, 80, 80, 40)`. The target is retained for error evaluation.

The cell below checks the stored arrays against the normalization formula and prints each channel's mean and standard deviation. These need not be exactly zero and one for this polymer because the scaling parameters are fixed across structures.


In [ ]:
subprocess.run([PYTHON, str(ROOT / 'tutorial/1.preprocess/3.collect.py'),
                str(WORK / 'raw'), str(SCALE),
                str(WORK / 'dataset')], check=True)

with h5py.File(WORK / 'dataset/polymer.h5', 'r') as dataset:
    feature = dataset['feature'][:]
    target = dataset['target'][:]
    print('HDF5 datasets:', list(dataset.keys()))
    print('feature:', feature.shape, '| target:', target.shape)

for i, channel in enumerate(channels[1:]):
    with h5py.File(WORK / 'raw' / channel / 'polymer.h5', 'r') as data:
        raw = data['value'][:]
    expected = (raw - scale[f'mean{i + 2}']) / scale[f'std{i + 2}']
    numpy.testing.assert_allclose(feature[i], expected)
    print(f'feature[{i}] ({channel}): mean={feature[i].mean():.6f}, '
          f'std={feature[i].std():.6f}')

with h5py.File(WORK / 'raw/target/polymer.h5', 'r') as data:
    numpy.testing.assert_array_equal(target, data['value'][:])
print('Verified: three normalized features and an unchanged SCF target.')


## 4. Predict the electron density
### Download the pretrained model
Download the pretrained archive and keep only `pretrained/r_plus_s_x3/model.pt`. The download is about 486 MB and is skipped if the weights are already present.

This tutorial uses inference only: neither the training dataset nor a training run is required.

In [ ]:
%%bash
set -euo pipefail
cd /content/DeepSCF
if [ ! -s pretrained/r_plus_s_x3/model.pt ]; then
  curl -fL --retry 3 https://github.com/yhkimlab/DeepSCF/releases/download/DeepSCF_v1/pretrained-models-v1.tar.gz \
    | tar --no-same-owner -xz pretrained/r_plus_s_x3/model.pt
fi
test -s pretrained/r_plus_s_x3/model.pt


### Run DeepSCF
Load the **(R+S)×3** model and predict the density on CPU. The helper writes `predict/polymer.RHO` using the same cell and grid as the reference calculation and checks the exported file.

Keep this predicted density for the next step; it will be passed to SIESTA as `In.RHO`.

In [ ]:
subprocess.run([PYTHON, str(SRC / 'predict_polymer.py'), str(ROOT), str(WORK)], check=True)
PREDICTED = WORK / 'predict/polymer.RHO'

### Compare the predicted and SCF densities
Compare the atomic, predicted, and reference densities on the same grid. The final panel shows the projected difference **DeepSCF − SCF**.

Use the full-grid relative errors to quantify the improvement over the atomic starting density. The difference plot helps locate the regions where the prediction differs from the reference.

In [ ]:
plot_density(SCF / 'Molecular.RHO', atomic=SCF / 'rhoatom', prediction=PREDICTED);

## 5. Non-SCF calculation
### Prepare and inspect the inputs
Use three cases at the **same fixed geometry**: the initial atomic density (ρ_init), the predicted density (ρ_ML), and the converged SCF reference (ρ_SCF). Run two separate one-step calculations, using `rhoatom` and the prediction as `In.RHO`, respectively. The reference is the converged calculation from Section 2.

Both one-step runs use the same `RUN.fdf`:
- `SCF.Read.Charge T` reads `In.RHO`.
- `MaxSCFIterations 1` performs one iteration.
- `SCF.Mix.First F` disables first-step mixing.
- `SCF.MustConverge F` permits completion without convergence.

The geometry, pseudopotentials, basis, grid, and k points are identical. `DM.UseSaveDM F` prevents reuse of a converged density matrix. `WriteForces T` prints the forces, and `Molecular.FA` stores atom indices and Fx, Fy, Fz in eV/Å.


In [ ]:
for folder, rho in [(INITIAL, SCF / 'rhoatom'), (NSCF, PREDICTED)]:
    shutil.copy(rho, folder / 'In.RHO')
    print('Density input:', rho, '->', folder / 'In.RHO')
print('--- shared one-step RUN.fdf ---')
print((NSCF / 'RUN.fdf').read_text())


### Run the one-step calculations
Calculate the electronic structure separately from ρ_init and ρ_ML. The helper verifies **exactly one SCF iteration** and checks that a force file was written for each run. An `SCF_NOT_CONV` message is expected.

These are actual SIESTA calculations, not quantities supplied by the neural network. SIESTA diagonalizes the Hamiltonian constructed using the input density, then uses the output density matrix (`DM_out`) for its final energy and forces. The table therefore compares **results of calculations started from each density**, rather than direct evaluations of E[ρ_init] or E[ρ_ML].


In [ ]:
for label, folder in [('rho_init', INITIAL), ('rho_ML', NSCF)]:
    print(label)
    subprocess.run([PYTHON, str(SRC / 'run_siesta.py'), str(folder),
                    '--siesta', str(SIESTA), '--non-scf'], check=True)


### Compare total energies and atomic forces
The table compares **ρ_init | ρ_ML | ρ_SCF**. Density errors refer to the supplied grids; energies and forces are read from the corresponding SIESTA outputs. All energy entries use the same final `siesta: Total` field.

Energy error is |E − E_SCF|/N_atom. Force component RMSE is sqrt[Σ_iα(F_iα − F_iα^SCF)²/(3N_atom)]; maximum force-vector error is max_i|F_i − F_i^SCF|. The structure is fixed, so the SCF forces need not vanish.

Compare the initial and ML columns to assess how much the prediction improves the subsequent calculation. Small density errors alone do not guarantee accurate forces. These non-self-consistent forces are a comparison with SCF, not a demonstration of conservative forces for ML-driven relaxation or molecular dynamics.


In [ ]:
from IPython.display import Markdown, display
summary = compare_observables(SCF, INITIAL, NSCF, PREDICTED)
display(Markdown(summary))

### Inspect the atomic forces
Compare force magnitudes for all three cases and the vector error relative to SCF for each atom. The error includes both direction and magnitude differences.

The vertical axes use a symmetric-log scale (linear below 0.001 eV/Å) so the smaller ML and SCF forces remain visible alongside the initial-density result.


In [ ]:
plot_forces(SCF, INITIAL, NSCF);

### Compare the band structures
Overlay the Γ–X bands for the same three cases. All curves use the reference SCF valence-band maximum as a common energy zero.


In [ ]:
plot_bands(SCF, NSCF, initial=INITIAL);

### Scope and output files
The fixed 0.25 Å grid follows the pretrained example; basis, mesh, and k-point convergence are not established here. The reference SCF is included for comparison, so the full workflow is not a speedup benchmark.

Logs: `stdout.txt` and `stderr.txt` in each run folder. Build log: `/content/siesta-build.log`.

[DeepSCF tutorial](https://github.com/yhkimlab/DeepSCF/blob/fe3c3c18052b51101e438dad2ffa06f94e77b5bb/tutorial/README.md) · [Polymer structure](https://github.com/yhkimlab/DeepSCF/blob/fe3c3c18052b51101e438dad2ffa06f94e77b5bb/tutorial/1.preprocess/structures/examples/polymer.fdf) · [sisl bands](https://sisl.readthedocs.io/en/latest/api/io/generated/sisl.io.siesta.bandsSileSiesta.html)
